## Simulation of $X$

For reliable and correct simulation of the SDE 
\begin{align}
dX_t & = (-X_t^3+ X_t- k)\,dt + \sigma_N \, dN_t^{(\alpha)} 
\end{align}
we use the direct splitting method recently developed in 

O. Aryasova, O. Kulyk, and I. Pavlyukevich. A tail-respecting explicit numerical scheme for
Lévy-driven SDEs with superlinear drifts. arXiv preprint arXiv:2504.07255, 2025

and 

I. Pavlyukevich, O. Aryasova, A. Chechkin, and O. Kulyk. How to simulate Lévy flights in a steep potential:
An explicit splitting numerical scheme. arXiv preprint arXiv:2508.07339, 2025

In [1]:
import numpy as np
import numpy.random as rng
import matplotlib.pyplot as plt
from scipy.stats import levy_stable, moment
from scipy.special import gamma
from scipy import linalg
from scipy.integrate import quad

import time

We simulate standard symmetric $\alpha$-stable random variables $\xi^{(\alpha)}$ with the ch.f.
\begin{align}
\mathbf E e^{iu\xi^{(\alpha)}}
=\begin{cases}
e^{-|u|^\alpha},\quad \alpha\in (0,2),\\
e^{-\frac{1}{2}u^2},\quad \alpha=2.
\end{cases}
\end{align}

We simulate the r.vs $\xi^{(\alpha)}$ directly, see function `stable_rv(alpha)` below.

Alternatively, for $\alpha=2$, we can use `standard_normal()`, and 
for $\alpha\in (0,2)$, we can use `levy_stable.rvs(alpha, 0)`.

Let $N^{(\alpha)}$ be a *standard* symmetric $\alpha$-stable L\'evy process with the ch.f.
\begin{align}
\mathbf E e^{iu N_t^{(\alpha)}}
=\begin{cases}
e^{-t|u|^\alpha},\quad \alpha\in (0,2),\\
e^{-\frac{t}{2}u^2},\quad \alpha=2.
\end{cases}
\end{align}
We set the scale parameter to be equal to 1, since the scale can be controlled later by the noise amplitude $\sigma_N$.


The increments of the noise of the amplitude $\sigma_N$:
$$
\sigma_N (N^{(\alpha)}_{nh} - N^{(\alpha)}_{(n-1)h})
\stackrel{d}{=} \sigma_N N^{(\alpha)}_h
\stackrel{d}{=} \sigma_N h^{1/\alpha}\xi^{(\alpha)}.
$$





To apply the (direct) splitting method, we single out the dynamics in the symmetric confining potential 
$$
U_0(x)=\frac{x^4}{4} - \frac{x^2}{2}.
$$

We solve the ODE 
\begin{align}
y'(t) &= -y(t)^3 + y(t),\\
y(0) & = x. 
\end{align}
Its solution is 
\begin{align}
\Phi(t,x)=\frac{x}{\sqrt{x^2(1-e^{-2t}) +e^{-2t}  }}  ,\quad x\in\mathbb R, \ t\in [0,\infty).
\end{align}

In [3]:
# the flow Phi(t,x)
# see (6.3)-(6.4)

def Phi(t,x):
    return x/np.sqrt( x*x*(1-np.exp(-2*t))  +  np.exp(-2*t)  )

The splitting numerical scheme $\{X^h_{nh}\}$.

Let 
\begin{align}
U(x,k)= \frac{x^4}{4} - \frac{x^2}{2} + kx.
\end{align}

Here we simulate the solutions of the SDE
\begin{align}
dX_t & = -U'(X_t,k)\,dt + \sigma_N\, dN_t^{(\alpha)} \\
dX_t & = (-X_t^3+ X_t- k)\,dt + \sigma_N \, dN_t^{(\alpha)} 
\end{align}
Parameters:
\begin{align}
&\alpha\in (0,2] \text{ - stability index}\\
&\sigma_N\in (0,\infty) \text{ - noise amplitude}\\
&k\in\mathbb R \text{ - asymmetry (bifurcation) parameter}\\
&T\in [0,\infty)\text{ - time interval}\\
&h\in (0,\infty)\text{ - time step}\\
&N=h^{-1}\in (0,\infty)\text{ - number of steps on the interval $[0,1]$}\\
&\text{we choose $T$ and $h$ such that $N, TN\in\mathbb N$}\\
&x\in\mathbb R\text{ - initial value}\\
\end{align}

We use the (direct) splitting method 
\begin{align}
\text{initial value }&& X^h_{0}&=x,\\
\text{intermediate step }&& Y^h_{nh}&=X_{(n-1)h}^h - k h + \sigma_N h^{1/\alpha}\xi_n^{(\alpha)},\\
\text{approximation }&& X_{nh}^h&= \Phi(h, Y^h_{nh}),\quad n=1,\dots,NT
\end{align}
or, explicitly,
\begin{align}
X^h_{0}&=x,\\
Y^h_{nh}&=X_{(n-1)h}^h - k h +\sigma_N h^{1/\alpha}\xi_n^{(\alpha)},\\
X_{nh}^h&= \frac{Y^h_{nh}}{\sqrt{(Y^h_{nh})^2(1-e^{-2h}) +e^{-2h}  }}  ,\quad n=1,\dots,NT
\end{align}

(naming convetions slightly adapted in following to match old code)

In [1]:
# external parameters
alpha = 0.9
k = -0.39

# simulation setting
Xzero = 1
N = 1000
dt = 1.0/N
T = 100
sigma = .6

X = np.zeros(T*N+1, dtype=float)

X[0] = Xzero

for n in range(1,T*N+1):
    xi = dL = stats.levy_stable.rvs(alpha=alpha, beta=0, loc=0, scale=1, random_state=1, size=1)
    y =  X[n-1] - k*dt + sigma*np.power(dt,1/alpha)*xi
    X[n] = Phi(dt, y)

plt.plot(np.linspace(0,T,T*N+1),X)
plt.ylim(-3, 3)
plt.show()

#print("time used = ",time.time()-start_time)

NameError: name 'np' is not defined

We loop over the different values of $\alpha$ and $k$ we want to simulate and calculate the rolling meand and variance for each trajectory to see when it stabilizes

In [ ]:
# external parameters
alphas = [2.0, 1.5, 1.0, 0.5]
ks = [0, -0.39, -1]

# simulation setting
num_simulations = 100  # number of samples in Monte Carlo

Xzero = 1.0  # initial value
T = 20        # we integrate solutions on the time interval [0,T]
N = 1000     # number of steps pro time unit
dt = 1.0/N   # time mesh

sigma = .6


# Create the figure and subplots
fig, axs = plt.subplots(2, 3, figsize=(15, 10))

for col_idx, k in enumerate(ks):
   
    for alpha in alphas:

        print(f'Starting: k = {k} and alpha = {alpha}')

        color = plt.cm.viridis(alpha / 2.0)

        noise_amplitude = sigma*np.power(dt,1/alpha)

        X = [Xzero]*num_simulations # SDE values
        
        empirical_mean = np.zeros(T*N+1, dtype=float) 
        empirical_var  = np.zeros(T*N+1, dtype=float)
        
        empirical_mean[0] = np.mean(X) 
        empirical_var[0]  = np.var(X)      
        
        for n in range(1,T*N+1):
            for j in range(num_simulations):             
                xi = xi = dL = stats.levy_stable.rvs(alpha=alpha, beta=0, loc=0, scale=1, random_state=j, size=1)
                y =  X[j] - k*dt + noise_amplitude*xi
                X[j] = Phi(dt, y)
                empirical_mean[n] = np.mean(X) # empirical mean     at time n*dt
                empirical_var[n]  = np.var(X)  # empirical variance at time n*dt

        axs[0, col_idx].plot(np.linspace(0, T, T * N + 1), empirical_mean, color=color)

        axs[1, col_idx].plot(np.linspace(0, T, T * N + 1), empirical_var, color=color)

        axs[0, col_idx].set_title(f'k = {k}')
        axs[0, col_idx].set_xlabel('Time')
        axs[0, col_idx].set_ylabel('Rolling Mean')
        axs[0, col_idx].grid(True)
        
        axs[1, col_idx].set_xlabel('Time')
        axs[1, col_idx].set_ylabel('Rolling Variance')
        axs[1, col_idx].grid(True)
        
handles, labels = axs[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right', title='Alpha Values')

plt.tight_layout(rect=[0, 0, 0.85, 1])
plt.show()         

Starting: k = 0 and alpha = 2.0
Starting: k = 0 and alpha = 1.5
Starting: k = 0 and alpha = 1.0
Starting: k = 0 and alpha = 0.5


Mean stabilized in monostable system, does not stabilize in metastable ssy
Variance stabilized for all systems